# Exercise 3 — Layered JSON Configuration with `ChainMap`

This standalone notebook solves Exercise #3 using `collections.ChainMap`.

The goal is to combine:
- `common.json` — shared/default settings;
- an environment-specific file such as `dev.json` or `prod.json`;

without duplicating the underlying mappings.

Environment-specific settings must override values from `common.json`.

This solution also adds:
- `pathlib`-based file handling;
- explicit environment-name validation;
- clear file and JSON error handling;
- validation that each JSON document contains an object/dictionary;
- immutable environment discovery;
- support for custom configuration directories;
- inspection of configuration layers;
- materialization to a regular dictionary when needed;
- tests using temporary JSON files;
- explanation of `ChainMap` lookup and mutation behavior;
- complexity and production-oriented notes.

## Problem

Suppose these files exist:

```text
common.json
dev.json
prod.json
```

We want a function such as:

```python
settings = load_settings("dev")
```

The result should behave like one combined mapping where values in `dev.json` override values from `common.json`.

The important requirement is to avoid copying both dictionaries into a third merged dictionary. `ChainMap` is designed for exactly this kind of layered lookup.

## Imports

In [1]:
from __future__ import annotations

import json
from collections import ChainMap
from collections.abc import Mapping
from pathlib import Path
from tempfile import TemporaryDirectory
from typing import Any

## Why `ChainMap` works

`ChainMap` groups multiple mappings and searches them from left to right.

Therefore:

```python
ChainMap(environment_settings, common_settings)
```

has exactly the lookup behavior we need:

1. look in the environment-specific mapping first;
2. if the key is absent, look in the common mapping.

The original dictionaries remain separate rather than being copied into a new merged dictionary.

## JSON loading helper

A small helper keeps file parsing and validation separate from configuration-layer logic.

In [2]:
JSONMapping = dict[str, Any]


def load_json_object(path: Path) -> JSONMapping:
    """Load a JSON file and require its top-level value to be an object.

    Parameters
    ----------
    path:
        Path to the JSON file.

    Returns
    -------
    dict[str, Any]
        Parsed JSON object.

    Raises
    ------
    FileNotFoundError
        If the file does not exist.
    json.JSONDecodeError
        If the file is not valid JSON.
    TypeError
        If the JSON document's top-level value is not an object.
    """
    with path.open("r", encoding="utf-8") as file:
        data = json.load(file)

    if not isinstance(data, dict):
        raise TypeError(
            f"Expected {path.name!r} to contain a JSON object, "
            f"got {type(data).__name__}."
        )

    return data

## Environment-name validation

The exercise assumes values such as `"dev"` and `"prod"`.

Because the environment name becomes part of a filename, validating it also prevents accidental path traversal such as `"../../some_file"`.

In [3]:
def validate_environment_name(environment: str) -> str:
    """Validate and normalize an environment identifier."""
    if not isinstance(environment, str):
        raise TypeError(
            f"environment must be a string, got {type(environment).__name__}."
        )

    environment = environment.strip()

    if not environment:
        raise ValueError("environment cannot be empty.")

    if not all(char.isalnum() or char in {"-", "_"} for char in environment):
        raise ValueError(
            "environment may contain only letters, numbers, '-' and '_'."
        )

    return environment

## Main solution — `load_settings`

The environment-specific dictionary is placed first in the `ChainMap`, so its keys take precedence over common defaults.

In [4]:
def load_settings(
    environment: str,
    *,
    config_dir: str | Path = ".",
) -> ChainMap[str, Any]:
    """Load common and environment-specific settings as a ``ChainMap``.

    Parameters
    ----------
    environment:
        Environment name, for example ``"dev"`` or ``"prod"``.
        The corresponding file is ``<environment>.json``.
    config_dir:
        Directory containing ``common.json`` and environment JSON files.
        Defaults to the current working directory.

    Returns
    -------
    ChainMap[str, Any]
        Layered configuration where environment-specific values override
        common values without copying both dictionaries into a merged dict.
    """
    environment = validate_environment_name(environment)
    directory = Path(config_dir).expanduser()

    common_path = directory / "common.json"
    environment_path = directory / f"{environment}.json"

    common_settings = load_json_object(common_path)
    environment_settings = load_json_object(environment_path)

    return ChainMap(environment_settings, common_settings)

## Using the provided exercise files

If `common.json`, `dev.json`, and `prod.json` are in the same directory as the notebook, usage is simply:

In [5]:
# Uncomment when the provided JSON files are available next to the notebook.
#
# dev_settings = load_settings("dev")
# prod_settings = load_settings("prod")
#
# print(dev_settings)
# print(prod_settings)

## Self-contained demonstration

The following example creates temporary configuration files so this notebook can demonstrate the behavior even without the original downloaded JSON files.

In [6]:
demo_common = {
    "host": "example.com",
    "port": 443,
    "debug": False,
    "log_level": "INFO",
    "timeout": 30,
}

demo_dev = {
    "host": "localhost",
    "port": 8000,
    "debug": True,
    "log_level": "DEBUG",
}

demo_prod = {
    "host": "api.example.com",
    "timeout": 10,
}

In [7]:
def write_json(path: Path, data: Mapping[str, Any]) -> None:
    """Write JSON test/demo data with readable formatting."""
    with path.open("w", encoding="utf-8") as file:
        json.dump(data, file, indent=2, ensure_ascii=False)
        file.write("\n")

In [8]:
with TemporaryDirectory() as temp_dir:
    config_dir = Path(temp_dir)

    write_json(config_dir / "common.json", demo_common)
    write_json(config_dir / "dev.json", demo_dev)
    write_json(config_dir / "prod.json", demo_prod)

    dev_settings = load_settings("dev", config_dir=config_dir)
    prod_settings = load_settings("prod", config_dir=config_dir)

    print("Development:")
    print(dict(dev_settings))

    print("\nProduction:")
    print(dict(prod_settings))

Development:
{'host': 'localhost', 'port': 8000, 'debug': True, 'log_level': 'DEBUG', 'timeout': 30}

Production:
{'host': 'api.example.com', 'port': 443, 'debug': False, 'log_level': 'INFO', 'timeout': 10}


## Override behavior

For the development environment:

- `host`, `port`, `debug`, and `log_level` come from `dev.json`;
- `timeout` falls back to `common.json` because it is not defined in `dev.json`.

In [9]:
with TemporaryDirectory() as temp_dir:
    config_dir = Path(temp_dir)

    write_json(config_dir / "common.json", demo_common)
    write_json(config_dir / "dev.json", demo_dev)

    settings = load_settings("dev", config_dir=config_dir)

    assert settings["host"] == "localhost"       # overridden
    assert settings["port"] == 8000              # overridden
    assert settings["debug"] is True             # overridden
    assert settings["log_level"] == "DEBUG"     # overridden
    assert settings["timeout"] == 30              # common fallback

    print("host:     ", settings["host"])
    print("port:     ", settings["port"])
    print("debug:    ", settings["debug"])
    print("log_level:", settings["log_level"])
    print("timeout:  ", settings["timeout"])

host:      localhost
port:      8000
debug:     True
log_level: DEBUG
timeout:   30


## Inspecting the underlying layers

`ChainMap.maps` exposes the mappings in lookup order.

For our implementation:

```python
settings.maps[0]  # environment-specific settings
settings.maps[1]  # common settings
```

In [10]:
with TemporaryDirectory() as temp_dir:
    config_dir = Path(temp_dir)

    write_json(config_dir / "common.json", demo_common)
    write_json(config_dir / "dev.json", demo_dev)

    settings = load_settings("dev", config_dir=config_dir)

    environment_layer, common_layer = settings.maps

    print("Environment layer:")
    print(environment_layer)

    print("\nCommon layer:")
    print(common_layer)

Environment layer:
{'host': 'localhost', 'port': 8000, 'debug': True, 'log_level': 'DEBUG'}

Common layer:
{'host': 'example.com', 'port': 443, 'debug': False, 'log_level': 'INFO', 'timeout': 30}


## Proving that no merged dictionary is created

`ChainMap` stores references to the original mappings. It does not eagerly copy all key/value pairs into another dictionary.

We can demonstrate that changes to an underlying mapping are immediately visible through the `ChainMap`.

In [11]:
common = {
    "timeout": 30,
    "debug": False,
}

development = {
    "debug": True,
}

settings = ChainMap(development, common)

print("Before change:", settings["timeout"])

common["timeout"] = 60

print("After change: ", settings["timeout"])

assert settings["timeout"] == 60

Before change: 30
After change:  60


## Important `ChainMap` mutation behavior

Assignments through a `ChainMap` always affect its **first mapping**.

For example:

In [12]:
common = {"timeout": 30, "debug": False}
development = {"debug": True}

settings = ChainMap(development, common)

settings["timeout"] = 5

print("Environment layer:", development)
print("Common layer:     ", common)
print("Effective value:  ", settings["timeout"])

assert development["timeout"] == 5
assert common["timeout"] == 30

Environment layer: {'debug': True, 'timeout': 5}
Common layer:      {'timeout': 30, 'debug': False}
Effective value:   5


This behavior is often useful: runtime overrides can naturally be written into the highest-priority layer without modifying common defaults.

## Converting to a regular dictionary

Sometimes another API specifically requires a real `dict`. In that case, materialize the effective configuration explicitly:

```python
resolved = dict(settings)
```

This **does create a copy**, so it should only be done when needed.

In [13]:
common = {"host": "example.com", "port": 443, "debug": False}
development = {"host": "localhost", "debug": True}

settings = ChainMap(development, common)
resolved = dict(settings)

print(resolved)

assert resolved == {
    "host": "localhost",
    "port": 443,
    "debug": True,
}

{'host': 'localhost', 'port': 443, 'debug': True}


## Discovering available environments

As a useful extension, we can inspect a configuration directory and return all environment files except `common.json`.

In [14]:
def discover_environments(config_dir: str | Path = ".") -> tuple[str, ...]:
    """Return available environment names in deterministic order."""
    directory = Path(config_dir).expanduser()

    environments = {
        path.stem
        for path in directory.glob("*.json")
        if path.name != "common.json" and path.is_file()
    }

    return tuple(sorted(environments))

In [15]:
with TemporaryDirectory() as temp_dir:
    config_dir = Path(temp_dir)

    write_json(config_dir / "common.json", {})
    write_json(config_dir / "dev.json", {})
    write_json(config_dir / "prod.json", {})
    write_json(config_dir / "staging.json", {})

    print(discover_environments(config_dir))

    assert discover_environments(config_dir) == (
        "dev",
        "prod",
        "staging",
    )

('dev', 'prod', 'staging')


## Optional advanced version — an additional runtime override layer

A major strength of `ChainMap` is that configuration can naturally have more than two layers.

For example:

```text
runtime overrides
        ↓
environment settings
        ↓
common defaults
```

The first mapping always has the highest priority.

In [16]:
def load_settings_with_overrides(
    environment: str,
    *,
    config_dir: str | Path = ".",
    overrides: Mapping[str, Any] | None = None,
) -> ChainMap[str, Any]:
    """Load configuration with an optional highest-priority override layer."""
    base_settings = load_settings(
        environment,
        config_dir=config_dir,
    )

    if overrides is None:
        return base_settings

    return ChainMap(dict(overrides), *base_settings.maps)

In [17]:
with TemporaryDirectory() as temp_dir:
    config_dir = Path(temp_dir)

    write_json(
        config_dir / "common.json",
        {"host": "example.com", "port": 443, "timeout": 30},
    )
    write_json(
        config_dir / "dev.json",
        {"host": "localhost", "port": 8000},
    )

    settings = load_settings_with_overrides(
        "dev",
        config_dir=config_dir,
        overrides={"port": 9000},
    )

    assert settings["host"] == "localhost"
    assert settings["port"] == 9000
    assert settings["timeout"] == 30

    print(dict(settings))

{'host': 'localhost', 'port': 9000, 'timeout': 30}


## Tests

The test suite uses temporary directories, so it does not depend on external files and leaves no configuration files behind.

In [18]:
def run_tests() -> None:
    with TemporaryDirectory() as temp_dir:
        config_dir = Path(temp_dir)

        common = {
            "host": "example.com",
            "port": 443,
            "debug": False,
            "timeout": 30,
        }

        dev = {
            "host": "localhost",
            "port": 8000,
            "debug": True,
        }

        prod = {
            "host": "api.example.com",
        }

        write_json(config_dir / "common.json", common)
        write_json(config_dir / "dev.json", dev)
        write_json(config_dir / "prod.json", prod)

        # -------------------------------------------------------------
        # Main behavior
        # -------------------------------------------------------------
        dev_settings = load_settings("dev", config_dir=config_dir)

        assert isinstance(dev_settings, ChainMap)
        assert len(dev_settings.maps) == 2

        # Environment-specific values override common values.
        assert dev_settings["host"] == "localhost"
        assert dev_settings["port"] == 8000
        assert dev_settings["debug"] is True

        # Missing environment values fall back to common settings.
        assert dev_settings["timeout"] == 30

        # Production gets its own overrides.
        prod_settings = load_settings("prod", config_dir=config_dir)
        assert prod_settings["host"] == "api.example.com"
        assert prod_settings["port"] == 443
        assert prod_settings["debug"] is False

        # -------------------------------------------------------------
        # Materialized result
        # -------------------------------------------------------------
        assert dict(dev_settings) == {
            "host": "localhost",
            "port": 8000,
            "debug": True,
            "timeout": 30,
        }

        # -------------------------------------------------------------
        # Layer ordering
        # -------------------------------------------------------------
        assert dev_settings.maps[0] == dev
        assert dev_settings.maps[1] == common

        # -------------------------------------------------------------
        # Environment discovery
        # -------------------------------------------------------------
        assert discover_environments(config_dir) == ("dev", "prod")

        # -------------------------------------------------------------
        # Runtime overrides
        # -------------------------------------------------------------
        overridden = load_settings_with_overrides(
            "dev",
            config_dir=config_dir,
            overrides={"port": 9999, "feature_x": True},
        )

        assert len(overridden.maps) == 3
        assert overridden["port"] == 9999
        assert overridden["feature_x"] is True
        assert overridden["timeout"] == 30

        # -------------------------------------------------------------
        # Missing environment file
        # -------------------------------------------------------------
        try:
            load_settings("staging", config_dir=config_dir)
        except FileNotFoundError:
            pass
        else:
            raise AssertionError(
                "Missing environment files should raise FileNotFoundError."
            )

        # -------------------------------------------------------------
        # Invalid environment names
        # -------------------------------------------------------------
        for invalid_environment in ("", "   ", "../prod", "dev/test"):
            try:
                load_settings(
                    invalid_environment,
                    config_dir=config_dir,
                )
            except ValueError:
                pass
            else:
                raise AssertionError(
                    f"Expected {invalid_environment!r} to be rejected."
                )

        # -------------------------------------------------------------
        # JSON top-level type validation
        # -------------------------------------------------------------
        bad_path = config_dir / "bad.json"
        bad_path.write_text('[1, 2, 3]', encoding="utf-8")

        try:
            load_settings("bad", config_dir=config_dir)
        except TypeError:
            pass
        else:
            raise AssertionError(
                "A top-level JSON array should be rejected."
            )

        # -------------------------------------------------------------
        # Invalid JSON
        # -------------------------------------------------------------
        invalid_json_path = config_dir / "broken.json"
        invalid_json_path.write_text('{not valid json}', encoding="utf-8")

        try:
            load_settings("broken", config_dir=config_dir)
        except json.JSONDecodeError:
            pass
        else:
            raise AssertionError(
                "Invalid JSON should raise JSONDecodeError."
            )

    print("All tests passed.")


run_tests()

All tests passed.


## Complexity

Let:

- `E` = number of keys in the environment-specific dictionary;
- `C` = number of keys in the common dictionary.

Parsing the JSON files necessarily takes approximately **O(E + C)** time and space because both files must be loaded into memory.

Creating the `ChainMap` itself is **O(1)**: it stores references to the existing mappings rather than copying all their entries.

For two layers, a lookup is effectively **O(1)** expected time because dictionary lookups are expected O(1), with at most two dictionaries searched.

Calling:

```python
dict(settings)
```

materializes the effective configuration and therefore requires **O(U)** additional time and space, where `U` is the number of unique configuration keys.

## Best-practice notes

### 1. Put the highest-priority mapping first

Correct:

```python
ChainMap(environment_settings, common_settings)
```

Incorrect for this exercise:

```python
ChainMap(common_settings, environment_settings)
```

The latter would cause common values to override environment-specific ones.

### 2. Do not immediately convert to `dict`

Doing this:

```python
dict(ChainMap(environment_settings, common_settings))
```

works functionally, but defeats the exercise's requirement to avoid a duplicated merged dictionary.

### 3. Keep parsing separate from precedence logic

`load_json_object()` handles file concerns, while `load_settings()` handles configuration layering. This makes both pieces easier to test and reuse.

### 4. Validate external configuration

Configuration files are external input. Invalid JSON, unexpected top-level arrays, missing files, and unsafe environment names should fail clearly rather than producing obscure downstream errors.

## Minimal exercise answer

If the exercise requires only the essential implementation and all inputs are trusted, the core solution can be reduced to:

In [19]:
def load_settings_minimal(environment: str) -> ChainMap:
    with open("common.json", encoding="utf-8") as file:
        common = json.load(file)

    with open(f"{environment}.json", encoding="utf-8") as file:
        environment_specific = json.load(file)

    return ChainMap(environment_specific, common)

## Recommendation

The essential idea for Exercise #3 is:

```python
return ChainMap(environment_settings, common_settings)
```

Placing the environment-specific mapping first gives it higher priority while keeping both source dictionaries separate.

For reusable application code, prefer the full `load_settings()` implementation because it adds safer path handling, validation, clearer errors, and support for a configurable directory without changing the fundamental `ChainMap` solution.